# Predictive Maintenance — Random Forest Classifier
### AI4I 2020 Predictive Maintenance Dataset

Predicts **Machine failure** from mechanical sensor readings: Air temperature, Process temperature, Rotational speed, Torque, and Tool wear.

**If running in Google Colab:** upload `ai4i2020.csv` using the file icon on the left sidebar (or mount Google Drive) before running the cells below.


In [ ]:
# If a package is missing (e.g. in a fresh Colab runtime), uncomment:
# !pip install pandas scikit-learn matplotlib joblib

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
)

## 1. Load the data

In [ ]:
df = pd.read_csv("ai4i2020.csv")

FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
TARGET = "Machine failure"

X = df[FEATURES]
y = df[TARGET]

print(f"Dataset shape: {df.shape}")
print(f"Failure rate: {y.mean():.2%}  ({y.sum()} failures out of {len(y)} records)")
df.head()

**Note on leakage:** the dataset also includes `TWF`, `HDF`, `PWF`, `OSF`, `RNF` columns. These record *which* failure mode occurred, so they're only known after a failure already happened — they are deliberately excluded from `FEATURES` above. Including them would let the model "cheat" and make the feature importances meaningless.

## 2. Train / test split

`stratify=y` keeps the ~3.4% failure ratio consistent in both splits, since failures are rare and a plain random split could easily under- or over-sample them.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 3. Train the Random Forest

`class_weight="balanced"` tells the forest to weight the rare failure class more heavily, so it doesn't just learn to always predict "no failure" (which would already be ~97% "accurate" but useless).

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## 4. Evaluate on the held-out test set

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No failure", "Failure"]))

## 5. Feature importance — which mechanical factor matters most?

In [ ]:
importances = (
    pd.Series(model.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)
print(importances)

plt.figure(figsize=(8, 5))
importances.sort_values().plot(kind="barh", color="#4C72B0")
plt.xlabel("Importance")
plt.title("What drives predicted machine failure?")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

## 6. Save the trained model

This writes `model.pkl`, which the Streamlit app (`app.py`) loads to serve predictions.

In [ ]:
joblib.dump(model, "model.pkl")
print("Saved trained model to model.pkl")